# CS 3892 / 5892 — Session 17 · Bounded model checking of programs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-27-bounded-model-checking-of-programs.ipynb)

**Tuesday, October 27, 2026.** A C program is a transition system over its memory states. **CBMC**
unrolls its loops up to a bound, turns the unrolled program into one formula, and asks a SAT solver
whether any assertion can fail: bounded model checking, as in session 16, with the program itself as
the model. Three programs: the running counter, session 11's agent, and a retry loop that shows what
an **unwinding assertion** is for.

The bounded and inductive checks on SMV models from the first half of class are in the
[session 16 notebook](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-20-symbolic-and-bounded-model-checking.ipynb).
Every file here asserts its own results. CBMC (free, BSD) is fetched by the setup cell.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, fetch CBMC, define helpers
import os, re, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-10-27-bounded-model-checking-of-programs"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

# Colab does not ship Z3; install it if this machine lacks it.
try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

PYD = ROOT / "sessions" / SESSION / "python"
CD  = ROOT / "sessions" / SESSION / "c"
FIG = ROOT / "sessions" / SESSION / "figures"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- CBMC: the official 6.11.0 release, fetched once (about 70 MB) and checked by SHA-256.
# cbmc_cli.py unpacks the Ubuntu 22.04 package in Python: no root, no dpkg needed.
out = run(PYD / "cbmc_cli.py")
assert "CBMC 6.11.0" in out

## 1. A program is a transition system

The running counter, as a loop. A state is (the line the program is at, the value of `x`).
CBMC checks `x <= 10` after every one of 30 steps.

In [ ]:
display(SVG(filename=str(FIG / "counter.svg")))
show(CD / "counter.c")
out = run(PYD / "01_counter_in_c.py")
assert "VERIFICATION SUCCESSFUL" in out

## 2. Reading a counterexample: the agent, in C

`nondet_int()` has no body, so CBMC may make it return anything: the model's choice of action, and
the human's answer. The trace is the list of values that makes the assertion fail.

In [ ]:
display(SVG(filename=str(FIG / "agent_c.svg")))
show(CD / "agent.c")
out = run(PYD / "02_agent_trace.py")
assert "assertion approved: FAILURE" in out and "VERIFICATION SUCCESSFUL" in out

## 3. What an unwinding assertion protects you from

`--unwind k` looks at runs that leave each loop within *k* iterations. The **unwinding assertion**
(on by default since CBMC 6) checks that no run is longer than that. Turn it off and a too-small
bound gives a confident, wrong answer.

In [ ]:
display(SVG(filename=str(FIG / "retry.svg")))
show(CD / "retry.c")
out = run(PYD / "03_unwinding_assertions.py")
assert "unwinding assertion loop 0: FAILURE" in out and "counterexample: max_retries" in out

## Try it yourself

- In `retry.c`, change the claim to `attempts <= 5`. Which bound is the smallest that proves it, and
  what does CBMC say at the bound below that?
- In `agent.c`, delete the line `approved = 0;` after a delete. Does the property still hold with
  `-DGATEWAY`? What does that say about one approval and several deletes?
- Make the counter's cap a `nondet_int()` between 1 and 20. Which `--unwind` do you need now?